In [1]:
import pandas as pd
import random
import numpy as np
import datetime

In [2]:
def get_win_probabilities(team1, team2, rankings):
    team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
    team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
    total = team1_total + team2_total
    team1_prob = 1 - (team1_total * 1.0 / total)
    team2_prob = 1 - (team2_total * 1.0 / total)
    return team1_prob, team2_prob

In [3]:
dartboard_numbers = [20, 1, 18, 4, 13, 6, 10, 15, 2, 17, 3, 19, 7, 16, 8, 11, 14, 9, 12, 5]

# Indices for special positions
vertical_indices = [0, 10]      # 20 (top), 3 (bottom)
horizontal_indices = [5, 15]    # 6 (right), 11 (left)

# Build weights: 6 down to 1, then up to 6
descending = [7, 6, 5, 5, 4, 3]
ascending = [4, 5, 5, 6]
weights_first_half = descending + ascending  # 6,5,4,3,2,1,2,3,4,5 (10 numbers)
#weights_second_half = weights_first_half[::-1]  # Mirror for the second half

dartboard_weights_list = weights_first_half + weights_first_half  # 20 numbers

# Map weights to numbers
dartboard_weights = {num: weight for num, weight in zip(dartboard_numbers, dartboard_weights_list)}

# For reference, print the mapping
for num in dartboard_numbers:
    print(f"{num}: {dartboard_weights[num]}")

20: 7
1: 6
18: 5
4: 5
13: 4
6: 3
10: 4
15: 5
2: 5
17: 6
3: 7
19: 6
7: 5
16: 5
8: 4
11: 3
14: 4
9: 5
12: 5
5: 6


In [4]:
def get_dartboard_numbers(team1, team2, rankings, dartboard_numbers, dartboard_weights,
                          n_trials=2000, temperature=0.35, seed=None, return_debug=False):
    if seed is not None:
        random.seed(seed)

    # Probabilities from your function
    team1_prob, team2_prob = get_win_probabilities(team1, team2, rankings)

    total_weight = sum(dartboard_weights[n] for n in dartboard_numbers)
    team1_target = total_weight * team1_prob

    best = None  # (score, assignments_dict, team1_weight, team2_weight)

    for _ in range(n_trials):
        nums = dartboard_numbers[:]
        random.shuffle(nums)

        assignments = {}
        w1 = 0
        w2 = 0

        for n in nums:
            w = dartboard_weights[n]

            # Remaining "need" to hit the target (positive means still needs weight)
            need1 = team1_target - w1
            need2 = (total_weight - team1_target) - w2

            # Soft preference for the team that needs more weight
            # Convert needs into a probability using a squashed function
            # temperature controls randomness: smaller => more deterministic
            x = (need1 - need2) / max(1.0, temperature * total_weight)
            p_team1 = 1.0 / (1.0 + (2.718281828 ** (-x)))

            if random.random() < p_team1:
                assignments[n] = team1
                w1 += w
            else:
                assignments[n] = team2
                w2 += w

        # Score: closeness to team1_target (team2 is implied since totals sum)
        score = abs(w1 - team1_target)

        # Keep best (tie-break with a little randomness so repeated runs vary)
        if best is None or score < best[0] or (score == best[0] and random.random() < 0.5):
            best = (score, assignments, w1, w2)

    _, assignments, w1, w2 = best

    team_list = pd.Series(assignments).sort_index()

    if return_debug:
        return team_list, {
            "team1_prob": team1_prob,
            "team2_prob": team2_prob,
            "total_weight": total_weight,
            "team1_target": team1_target,
            "team1_weight": w1,
            "team2_weight": w2,
            "abs_error": abs(w1 - team1_target),
        }

    return team_list

In [5]:
# NOTE FOR 2026 - Weight the vertical numbers more than the horizontal numbers
def get_dartboard_numbers_old(team1, team2):
    team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
    team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
    total = team1_total + team2_total
    
    team1_prob = 1 - (team1_total * 1.0 / total)
    team2_prob = 1 - (team2_total * 1.0 / total)
    
    team1_size = team1_prob * 20
    team2_size = team2_prob * 20
    team1_size = round(team1_size,0)
    team2_size = round(team2_size,0)
    
    team_list = [team1] * int(team1_size) + [team2] * int(team2_size)
    
    num_shuffles = random.randint(1,50)
    for i in range(1,num_shuffles):
        random.shuffle(team_list)
        
    team_list = pd.Series(team_list)
    team_list.index += 1
    
    print (team_list)

In [6]:
def select_winner(team1, team2, rankings):
    # Probabilities from your function
    team1_prob, team2_prob = get_win_probabilities(team1, team2, rankings)
    
    team1_size = team1_prob * 100
    team2_size = team2_prob * 100
    team1_size = round(team1_size,0)
    team2_size = round(team2_size,0)
    
    team_list = [team1] * int(team1_size) + [team2] * int(team2_size)
    
    num_shuffles = random.randint(1,50)
    for i in range(1,num_shuffles):
        random.shuffle(team_list)
        
    team_list = pd.Series(team_list)
    team_list.index += 1
    
    return team_list[1]

In [7]:
rankings = pd.read_csv('CBB Rankings - 2026.csv')
rankings.set_index('Team', inplace=True)

In [68]:
# rankings.drop('Recent Rank', axis=1, inplace=True)

In [8]:
rankings.head(10)

,Bracket Order,Seed,Ken Pom,Off Eff,Def Eff,NET Ranking
Team,,,,,,
Duke,1,1,1,4,2,1
Siena,2,16,192,210,175,183
Ohio St.,3,8,26,17,54,29
TCU,4,9,43,81,22,39
St. John's,5,5,17,44,12,16
Northern Iowa,6,12,72,153,24,72
Kansas,7,4,21,57,10,21
Cal Baptist,8,13,106,191,50,98
Louisville,9,6,19,19,25,17


In [9]:
rankings.loc["Duke"]

Bracket Order    1
Seed             1
Ken Pom          1
Off Eff          4
Def Eff          2
NET Ranking      1
Name: Duke, dtype: int64

In [12]:
rankings.index

Index(['Duke', 'Siena', 'Ohio St.', 'TCU', 'St. John's', 'Northern Iowa',
       'Kansas', 'Cal Baptist', 'Louisville', 'South Florida', 'Michigan St.',
       'North Dakota St.', 'UCLA', 'UCF', 'Uconn', 'Furman', 'Florida',
       'Prairie View A&M', 'Clemson', 'Iowa', 'Vanderbilt', 'McNeese',
       'Nebraska', 'Troy', 'North Carolina', 'VCU', 'Illinois', 'Penn',
       'Saint Mary's', 'Texas A&M', 'Houston', 'Idaho', 'Arizona', 'LIU',
       'Villanova', 'Utah St.', 'Wisconsin', 'High Point', 'Arkansas',
       'Hawaii', 'BYU', 'Texas', 'Gonzaga', 'Kennesaw St.', 'Miami (FL)',
       'Missouri', 'Purdue', 'Queens', 'Michigan', 'Howard', 'Georgia',
       'Saint Louis', 'Texas Tech', 'Akron', 'Alabama', 'Hofstra', 'Tennessee',
       'SMU', 'Virginia', 'Wright St.', 'Kentucky', 'Santa Clara', 'Iowa St.',
       'Tennessee St.'],
      dtype='object', name='Team')

In [13]:
bracket = pd.DataFrame(rankings.index)
bracket.index += 1
bracket = bracket[:64]
bracket.head()

,Team
1,Duke
2,Siena
3,Ohio St.
4,TCU
5,St. John's


In [14]:
for i in range(1,len(bracket),2):
    team1 = bracket['Team'][i]
    team2 = bracket['Team'][i+1]
    team1_prob, team2_prob = get_win_probabilities(team1, team2, rankings)
    print ()
    print (team1, ' (',f"{team1_prob:.1%}", ') vs. ', team2, ' (',f"{team2_prob:.1%}",')')
    print (get_dartboard_numbers(team1, team2, rankings, dartboard_numbers, dartboard_weights))
    print ()

C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]



Duke  ( 98.6% ) vs.  Siena  ( 1.4% )
1     Duke
2     Duke
3     Duke
4     Duke
5     Duke
6     Duke
7     Duke
8     Duke
9     Duke
10    Duke
11    Duke
12    Duke
13    Duke
14    Duke
15    Duke
16    Duke
17    Duke
18    Duke
19    Duke
20    Duke
dtype: object


Ohio St.  ( 58.3% ) vs.  TCU  ( 41.7% )
1     Ohio St.
2          TCU
3          TCU
4     Ohio St.
5          TCU
6          TCU
7     Ohio St.
8     Ohio St.
9     Ohio St.
10         TCU
11         TCU
12    Ohio St.
13         TCU
14    Ohio St.
15         TCU
16    Ohio St.
17    Ohio St.
18         TCU
19    Ohio St.
20    Ohio St.
dtype: object


St. John's  ( 77.2% ) vs.  Northern Iowa  ( 22.8% )
1     Northern Iowa
2     Northern Iowa
3        St. John's
4        St. John's
5        St. John's
6     Northern Iowa
7     Northern Iowa
8        St. John's
9        St. John's
10    Northern Iowa
11       St. John's
12       St. John's
13       St. John's
14       St. John's
15       St. John's
16       St. John'

In [15]:
team1 = "Purdue"
team2 = 'Duke'
print (team1, ' vs. ', team2)
print (get_dartboard_numbers(team1, team2, rankings, dartboard_numbers, dartboard_weights))

Purdue  vs.  Duke
1       Duke
2       Duke
3       Duke
4       Duke
5       Duke
6     Purdue
7       Duke
8       Duke
9       Duke
10      Duke
11    Purdue
12      Duke
13      Duke
14    Purdue
15      Duke
16      Duke
17      Duke
18      Duke
19    Purdue
20      Duke
dtype: object


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]


In [16]:
team1 = 'Wisconsin'
team2 = 'Illinois'

print (get_win_probabilities(team1, team2,rankings))

print (select_winner(team1, team2,rankings))

(np.float64(0.30645161290322576), np.float64(0.6935483870967742))
Illinois


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]


In [17]:
second_round = []

for i in range(1,len(bracket),2):
    team1 = bracket['Team'][i]
    team2 = bracket['Team'][i+1]
    second_round.append(select_winner(team1, team2, rankings))

print (second_round)
print ()

sweet_16 = []

for i in range(1,len(second_round),2):
    team1 = second_round[i-1]
    team2 = second_round[i]
    sweet_16.append(select_winner(team1, team2, rankings))

print (sweet_16)
print ()

elite_8 = []

for i in range(1,len(sweet_16),2):
    team1 = sweet_16[i-1]
    team2 = sweet_16[i]
    elite_8.append(select_winner(team1, team2, rankings))

print (elite_8)
print ()

final_4 = []

for i in range(1,len(elite_8),2):
    team1 = elite_8[i-1]
    team2 = elite_8[i]
    final_4.append(select_winner(team1, team2, rankings))

print (final_4)
print ()

nat_champ = []

for i in range(1,len(final_4),2):
    team1 = final_4[i-1]
    team2 = final_4[i]
    nat_champ.append(select_winner(team1, team2, rankings))

print (nat_champ)
print ()

team1 = nat_champ[0]
team2 = nat_champ[1]
winner = select_winner(team1, team2, rankings)
print (winner)



['Duke', 'TCU', 'Northern Iowa', 'Kansas', 'Louisville', 'Michigan St.', 'UCLA', 'Uconn', 'Florida', 'Iowa', 'Vanderbilt', 'Nebraska', 'North Carolina', 'Illinois', 'Texas A&M', 'Houston', 'Arizona', 'Utah St.', 'Wisconsin', 'Arkansas', 'BYU', 'Gonzaga', 'Miami (FL)', 'Purdue', 'Michigan', 'Saint Louis', 'Texas Tech', 'Alabama', 'SMU', 'Virginia', 'Kentucky', 'Iowa St.']

['Duke', 'Kansas', 'Michigan St.', 'Uconn', 'Florida', 'Vanderbilt', 'Illinois', 'Houston', 'Arizona', 'Wisconsin', 'Gonzaga', 'Miami (FL)', 'Saint Louis', 'Texas Tech', 'Virginia', 'Kentucky']

['Kansas', 'Uconn', 'Florida', 'Houston', 'Arizona', 'Gonzaga', 'Texas Tech', 'Kentucky']

['Kansas', 'Houston', 'Arizona', 'Texas Tech']

['Houston', 'Arizona']

Houston


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]


In [18]:
all_winners = []


In [19]:
test.columns = ["Duke",
"Ohio St.",
"St. John's",
"Kansas",
"Louisville",
"Michigan St.",
"UCLA",
"Uconn",
"Florida",
"Clemson",
"Vanderbilt",
"Nebraska",
"North Carolina",
"Illinois",
"Saint Mary's",
"Houston",
"Arizona",
"Villanova",
"Wisconsin",
"Arkansas",
"BYU",
"Gonzaga",
"Miami (FL)",
"Purdue",
"Michigan",
"Georgia",
"Texas Tech",
"Alabama",
"Tennessee",
"Virginia",
"Kentucky",
"Iowa St.",
"East Sweet 16 1",
"East Sweet 16 2",
"East Sweet 16 3",
"East Sweet 16 4",
"West Sweet 16 1",
"West Sweet 16 2",
"West Sweet 16 3",
"West Sweet 16 4",
"South Sweet 16 1",
"South Sweet 16 2",
"South Sweet 16 3",
"South Sweet 16 4",
"Midwest Sweet 16 1",
"Midwest Sweet 16 2",
"Midwest Sweet 16 3",
"Midwest Sweet 16 4",
"East Elite 8 1",
"East Elite 8 2",
"West Elite 8 1",
"West Elite 8 2",
"South Elite 8 1",
"South Elite 8 2",
"Midwest Elite 8 1",
"Midwest Elite 8 2",
"East Champ",
"West Champ",
"South Champ",
"Midwest Champ",
"Nat Champ 1",
"Nat Champ 2",
"Champion"
]

NameError: name 'test' is not defined

In [27]:
test

,Alabama,Maryland,SDSU,Virginia,Creighton,Baylor,Missouri,Arizona,Purdue,Memphis,...,Midwest Elite 8 2,West Elite 8 1,West Elite 8 2,South Champ,East Champ,Midwest Champ,West Champ,Nat Champ 1,Nat Champ 2,Champion
0,Alabama,W. Virginia,SDSU,Virginia,Creighton,Baylor,Missouri,Arizona,Purdue,Memphis,...,Texas,St. Mary's,UCLA,Alabama,Marquette,Houston,UCLA,Alabama,Houston,Houston


In [32]:
j % 1000

NameError: name 'j' is not defined

In [21]:
winners = []

for j in range(0,10000):
    second_round = []

    for i in range(1,len(bracket),2):
        team1 = bracket['Team'][i]
        team2 = bracket['Team'][i+1]
        second_round.append(select_winner(team1, team2, rankings))

    sweet_16 = []

    for i in range(1,len(second_round),2):
        team1 = second_round[i-1]
        team2 = second_round[i]
        sweet_16.append(select_winner(team1, team2, rankings))

    elite_8 = []

    for i in range(1,len(sweet_16),2):
        team1 = sweet_16[i-1]
        team2 = sweet_16[i]
        elite_8.append(select_winner(team1, team2, rankings))

    final_4 = []

    for i in range(1,len(elite_8),2):
        team1 = elite_8[i-1]
        team2 = elite_8[i]
        final_4.append(select_winner(team1, team2, rankings))

    nat_champ = []

    for i in range(1,len(final_4),2):
        team1 = final_4[i-1]
        team2 = final_4[i]
        nat_champ.append(select_winner(team1, team2, rankings))

    team1 = nat_champ[0]
    team2 = nat_champ[1]
    winner = select_winner(team1, team2, rankings)
    winners.append(winner)
    
    temp = pd.DataFrame(second_round + sweet_16 + elite_8 + final_4 + nat_champ + [winner]).T
    if j == 0:
        brackets2 = temp
    else:
        brackets2 = pd.concat([brackets2, temp], ignore_index=True)
        
    if (j % 1000) == 0:
        print (j, datetime.datetime.now())
        
    #brackets2 = brackets2.append(temp)

C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as posi

0 2026-03-19 14:04:48.676160


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as posi

1000 2026-03-19 14:06:08.055912


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as posi

...

C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as posi

8000 2026-03-19 14:39:56.626110


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as posi

9000 2026-03-19 14:41:03.037403


C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team1_total = rankings.loc[team1][1] * 4 + rankings.loc[team1][2] + rankings.loc[team1][3] + rankings.loc[team1][4] + rankings.loc[team1][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team2_total = rankings.loc[team2][1] * 4 + rankings.loc[team2][2] + rankings.loc[team2][3] + rankings.loc[team2][4] + rankings.loc[team2][5]
C:\Users\maasr\AppData\Local\Temp\ipykernel_944\1088193208.py:2: FutureWarning: Series.__getitem__ treating keys as posi

In [22]:
brackets2[62].value_counts()

62
Duke              3764
Arizona           2092
Michigan          1866
Florida            862
Houston            385
Iowa St.           275
Purdue             109
Illinois           105
Uconn               95
Michigan St.        83
Gonzaga             81
Virginia            49
Vanderbilt          41
Arkansas            28
St. John's          18
Texas Tech          18
Wisconsin           18
Louisville          17
Nebraska            17
Tennessee           15
Kansas              11
Alabama             11
BYU                  9
Saint Mary's         5
North Carolina       4
Utah St.             4
Kentucky             3
UCLA                 3
South Florida        2
Miami (FL)           2
Ohio St.             2
Iowa                 2
SMU                  1
Clemson              1
Akron                1
Santa Clara          1
Name: count, dtype: int64

In [23]:
brackets2.columns = ["Duke",
"Ohio St.",
"St. John's",
"Kansas",
"Louisville",
"Michigan St.",
"UCLA",
"Uconn",
"Florida",
"Clemson",
"Vanderbilt",
"Nebraska",
"North Carolina",
"Illinois",
"Saint Mary's",
"Houston",
"Arizona",
"Villanova",
"Wisconsin",
"Arkansas",
"BYU",
"Gonzaga",
"Miami (FL)",
"Purdue",
"Michigan",
"Georgia",
"Texas Tech",
"Alabama",
"Tennessee",
"Virginia",
"Kentucky",
"Iowa St.",
"East Sweet 16 1",
"East Sweet 16 2",
"East Sweet 16 3",
"East Sweet 16 4",
"West Sweet 16 1",
"West Sweet 16 2",
"West Sweet 16 3",
"West Sweet 16 4",
"South Sweet 16 1",
"South Sweet 16 2",
"South Sweet 16 3",
"South Sweet 16 4",
"Midwest Sweet 16 1",
"Midwest Sweet 16 2",
"Midwest Sweet 16 3",
"Midwest Sweet 16 4",
"East Elite 8 1",
"East Elite 8 2",
"West Elite 8 1",
"West Elite 8 2",
"South Elite 8 1",
"South Elite 8 2",
"Midwest Elite 8 1",
"Midwest Elite 8 2",
"East Champ",
"West Champ",
"South Champ",
"Midwest Champ",
"Nat Champ 1",
"Nat Champ 2",
"Champion"
]

In [24]:
brackets2.shape

(10000, 63)

In [25]:
brackets2.to_csv('Simulated_Brackets_10K_2026.csv')

In [26]:
for i in brackets2.columns.values:
    print (brackets2[i].value_counts())
    print ()
    print ()

Duke
Duke     9898
Siena     102
Name: count, dtype: int64


Ohio St.
Ohio St.    5893
TCU         4107
Name: count, dtype: int64


St. John's
St. John's       7727
Northern Iowa    2273
Name: count, dtype: int64


Kansas
Kansas         8036
Cal Baptist    1964
Name: count, dtype: int64


Louisville
Louisville       7133
South Florida    2867
Name: count, dtype: int64


Michigan St.
Michigan St.        8773
North Dakota St.    1227
Name: count, dtype: int64


UCLA
UCLA    6483
UCF     3517
Name: count, dtype: int64


Uconn
Uconn     9208
Furman     792
Name: count, dtype: int64


Florida
Florida             9796
Prairie View A&M     204
Name: count, dtype: int64


Clemson
Iowa       5401
Clemson    4599
Name: count, dtype: int64


Vanderbilt
Vanderbilt    7964
McNeese       2036
Name: count, dtype: int64


Nebraska
Nebraska    8680
Troy        1320
Name: count, dtype: int64


North Carolina
North Carolina    6214
VCU               3786
Name: count, dtype: int64


Illinois
Illinois    9